# Akan (Asante Twi) Text-to-Speech — Colab Training & Evaluation

This notebook runs the existing `akan-tts` VITS pipeline end-to-end: dataset verification → preprocessing → fine-tuning → unseen-text inference → objective evaluation → listener evaluation artifacts → submission package.

**Target variety:** Asante Twi (`aka`)

**Storage policy:** Google Drive is used only for persistent run checkpoints, samples, evaluation outputs, and the final package. Dataset downloads/cache and temporary preprocessing stay on Colab's temporary disk. The notebook checks available Drive space and keeps checkpoint retention limited to reduce storage use.

> Before running: Runtime → Change runtime type → GPU (T4/A100 preferred). Keep Drive mounted so checkpoints survive Colab disconnects. No external TTS API is used; speech is generated locally from the trained VITS checkpoint.


In [ ]:
# 0. Runtime, persistent storage, and disk-space checks
!nvidia-smi -L

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, os
PERSIST = Path('/content/drive/MyDrive/akan_tts')
PERSIST.mkdir(parents=True, exist_ok=True)

# Keep Hugging Face downloads/cache on Colab's temporary disk, NOT Google Drive.
HF_CACHE = Path('/content/hf_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE / 'hub')

usage = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive used: {usage.used/1024**3:.2f} GiB / {usage.total/1024**3:.2f} GiB')
print(f'Drive free: {usage.free/1024**3:.2f} GiB')
if usage.free < 2 * 1024**3:
    raise RuntimeError('Less than 2 GiB of Google Drive space is free. Delete/unneeded files before training to avoid an interrupted checkpoint save.')
print('Persistent experiment directory:', PERSIST)
print('Temporary Hugging Face cache:', HF_CACHE)


## 1. Get the existing project

The notebook can run from a cloned repository **or** directly from the ZIP you upload to Colab. This avoids making the experiment dependent on an unfinished GitHub URL.

In [ ]:
# Choose ONE source.
REPO_URL = ""  # Optional: e.g. https://github.com/<user>/akan-tts
ZIP_PATH = "/content/akan-tts.zip"  # Optional: upload the supplied ZIP to Colab first

import os, shutil, zipfile, subprocess, textwrap

if Path('/content/akan-tts').exists():
    PROJECT = Path('/content/akan-tts')
elif REPO_URL.strip():
    subprocess.run(['git','clone',REPO_URL,'/content/akan-tts'], check=True)
    PROJECT = Path('/content/akan-tts')
elif Path(ZIP_PATH).exists():
    extract_root = Path('/content/akan_tts_unpacked')
    if extract_root.exists(): shutil.rmtree(extract_root)
    extract_root.mkdir()
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(extract_root)
    candidates = list(extract_root.rglob('akan_tts_colab.ipynb'))
    if not candidates: raise FileNotFoundError('Could not find akan_tts_colab.ipynb in the ZIP.')
    PROJECT = candidates[0].parent
    shutil.copytree(PROJECT, '/content/akan-tts', dirs_exist_ok=True)
    PROJECT = Path('/content/akan-tts')
else:
    from google.colab import files
    print('Upload akan-tts.zip')
    uploaded = files.upload()
    zip_name = next((n for n in uploaded if n.endswith('.zip')), None)
    if not zip_name: raise FileNotFoundError('Please upload akan-tts.zip')
    ZIP_PATH = '/content/' + zip_name
    extract_root = Path('/content/akan_tts_unpacked'); extract_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(extract_root)
    candidates = list(extract_root.rglob('akan_tts_colab.ipynb'))
    if not candidates: raise FileNotFoundError('Could not find akan_tts_colab.ipynb in the ZIP.')
    shutil.copytree(candidates[0].parent, '/content/akan-tts', dirs_exist_ok=True)
    PROJECT = Path('/content/akan-tts')

%cd /content/akan-tts
print('Project:', PROJECT)
print('Files:', [p.name for p in PROJECT.iterdir()])

## 2. Install a reproducible Colab environment

The project uses the maintained `coqui-tts` fork. The installation is pinned to a known compatible range rather than allowing a future major `transformers` release to break the training stack.

In [ ]:
# Install project dependencies.
!python -m pip install -q -U pip
!pip install -q "coqui-tts>=0.24,<0.28" "transformers>=4.47,<5" "datasets>=2.18" jiwer librosa soundfile tqdm pandas numpy pyloudnorm huggingface_hub

import torch, transformers, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. Experiment configuration

The defaults retain the existing project's choices. **Do not claim a dataset duration, speaker identity, or license in the report until the verification cell below prints the actual values.**

In [ ]:
CONFIG = {
    'hf_dataset': 'google/WaxalNLP',
    'hf_config': 'twi_tts',
    'split': 'train',
    'audio_col': 'audio',
    'text_col': 'transcription',
    'speaker_col': None,
    'speaker_id': None,
    'max_hours': None,          # None = use all eligible data; set e.g. 1.0 only for a pilot
    'val_size': 30,
    'min_dur': 1.0,
    'max_dur': 15.0,
    'epochs': 300,
    'batch_size': 16,
    'lr': 1e-4,
    'seed': 1234,
}

# Data stays on Colab's temporary disk; only experiment artifacts persist to Drive.
DATA_DIR = Path('/content/akan_tts_data')
RUN_DIR = PERSIST / 'runs'
SAMPLE_DIR = PERSIST / 'samples'
EVAL_DIR = PERSIST / 'eval'
for d in (RUN_DIR, SAMPLE_DIR, EVAL_DIR): d.mkdir(parents=True, exist_ok=True)
print(CONFIG)
print('Temporary prepared dataset:', DATA_DIR)
print('Persistent run artifacts:', RUN_DIR)


## 4. Verify the WAXAL/Twi dataset before preprocessing

This cell records the dataset metadata available through Hugging Face and inspects the actual fields. If the dataset is gated, Colab will prompt for a Hugging Face token.

The assignment requires an openly licensed dataset and asks the report to state the exact dataset source, license, variety, audio duration, and preprocessing. Keep the printed evidence with your experiment.

In [ ]:
from datasets import load_dataset, load_dataset_builder

# Make sure cache configuration is in place before loading the dataset.
ds_info = load_dataset_builder(CONFIG['hf_dataset'], CONFIG['hf_config'], cache_dir=str(HF_CACHE))
print('Dataset:', CONFIG['hf_dataset'])
print('Config:', CONFIG['hf_config'])
print('Description:', getattr(ds_info.info, 'description', None))
print('License metadata:', getattr(ds_info.info, 'license', None))
print('Features:', ds_info.info.features)

# Inspect fields and example. The preparation script may load/cache the split again.
ds_preview = load_dataset(CONFIG['hf_dataset'], CONFIG['hf_config'], split=CONFIG['split'], cache_dir=str(HF_CACHE))
print('Rows:', len(ds_preview))
print('Columns:', ds_preview.column_names)
print('First example keys:', ds_preview[0].keys())
print('First transcription:', ds_preview[0].get(CONFIG['text_col']))
print('First audio metadata:', {k:v for k,v in ds_preview[0][CONFIG['audio_col']].items() if k != 'array'})
if CONFIG['speaker_col'] and CONFIG['speaker_col'] in ds_preview.column_names:
    speakers = ds_preview.unique(CONFIG['speaker_col'])
    print('Speaker count:', len(speakers))
    print('Speakers:', speakers[:20])
else:
    print('Speaker column not configured; verify the dataset card/metadata for speaker count and variety.')


## 5. Preprocess the corpus

This calls the existing `scripts/prepare_data.py`: Unicode/text normalization, `ɛ`/`ɔ` preservation, number expansion, audio resampling/mono conversion, silence trimming, peak normalization, duration/rate filtering, and train/validation split.

In [ ]:
import shlex, subprocess, json, shutil
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
args = [
    'python','scripts/prepare_data.py',
    '--hf_dataset', CONFIG['hf_dataset'],
    '--split', CONFIG['split'],
    '--audio_col', CONFIG['audio_col'],
    '--text_col', CONFIG['text_col'],
    '--out', str(DATA_DIR),
    '--val_size', str(CONFIG['val_size']),
    '--min_dur', str(CONFIG['min_dur']),
    '--max_dur', str(CONFIG['max_dur']),
    '--seed', str(CONFIG['seed']),
]
if CONFIG['hf_config']: args += ['--hf_config', CONFIG['hf_config']]
if CONFIG['speaker_col']: args += ['--speaker_col', CONFIG['speaker_col'], '--speaker_id', str(CONFIG['speaker_id'])]
if CONFIG['max_hours'] is not None: args += ['--max_hours', str(CONFIG['max_hours'])]
print('Running:', ' '.join(shlex.quote(str(x)) for x in args))
subprocess.run(args, check=True)
stats = json.loads((DATA_DIR/'stats.json').read_text(encoding='utf-8'))
print(json.dumps(stats, ensure_ascii=False, indent=2))
print('Prepared dataset disk usage:')
print(f'{sum(p.stat().st_size for p in DATA_DIR.rglob("*") if p.is_file())/1024**3:.2f} GiB')


In [ ]:
# Inspect the prepared metadata and listen to a few REAL training/validation examples.
import pandas as pd, random
train_meta = pd.read_csv(DATA_DIR/'metadata_train.csv', sep='|', header=None, names=['id','text','normalized'])
val_meta = pd.read_csv(DATA_DIR/'metadata_val.csv', sep='|', header=None, names=['id','text','normalized'])
print('Train:', len(train_meta), 'Validation:', len(val_meta))
print(train_meta.head())

from IPython.display import Audio, display
import soundfile as sf
for uid in val_meta['id'].head(3):
    print(uid, val_meta.loc[val_meta.id.eq(uid),'text'].iloc[0])
    display(Audio(filename=str(DATA_DIR/'wavs'/f'{uid}.wav')))

## 6. Fine-tune VITS

The existing training script is used. Checkpoints are written to Google Drive so a Colab disconnect does not destroy the run.

**Important:** a final submission requires a completed training run and a usable checkpoint; merely executing a few epochs is not sufficient.

In [ ]:
import subprocess, os, re, json, time, shlex

# Full configured run. Use CONFIG['epochs'] for the final experiment, not a one-epoch smoke test.
train_cmd = [
    'python','scripts/train.py',
    '--data', str(DATA_DIR),
    '--out', str(RUN_DIR),
    '--epochs', str(CONFIG['epochs']),
    '--batch_size', str(CONFIG['batch_size']),
    '--lr', str(CONFIG['lr']),
    '--seed', str(CONFIG['seed']),
]
print('Running:', ' '.join(map(shlex.quote, train_cmd)))
print('Drive free before training:', f'{shutil.disk_usage("/content/drive/MyDrive").free/1024**3:.2f} GiB')
result = subprocess.run(train_cmd, text=True)
if result.returncode != 0:
    raise RuntimeError('Training failed. Inspect the traceback above before retrying. Check Drive space and GPU RAM if the failure is storage/memory-related.')
print('Training process completed.')


### Resume training after a Colab disconnect

If training was interrupted, inspect the Drive run directory and resume from the latest checkpoint rather than starting over.

In [ ]:
# To resume, set RESUME_RUN to the exact training run folder containing its trainer state/checkpoints.
RESUME_RUN = ''  # e.g. '/content/drive/MyDrive/akan_tts/runs/akan_vits-...'
if RESUME_RUN:
    resume_cmd = ['python','scripts/train.py','--data',str(DATA_DIR),'--out',str(RUN_DIR),'--continue_path',RESUME_RUN,'--epochs',str(CONFIG['epochs']),'--batch_size',str(CONFIG['batch_size']),'--lr',str(CONFIG['lr']),'--seed',str(CONFIG['seed'])]
    print('Running resume command:', ' '.join(shlex.quote(str(x)) for x in resume_cmd))
    subprocess.run(resume_cmd, check=True)
else:
    print('No resume requested. If the previous run completed, continue to checkpoint discovery.')


## 7. Locate and validate the trained checkpoint

In [ ]:
from pathlib import Path
# Search the persistent run tree and also the conventional project run folder in case a prior run used it.
search_roots = [RUN_DIR, PROJECT / 'runs']
all_ckpts = []
for root in search_roots:
    if root.exists(): all_ckpts.extend(root.rglob('*.pth'))
all_ckpts = sorted({p.resolve() for p in all_ckpts if p.is_file()}, key=lambda p: p.stat().st_mtime, reverse=True)
print('Checkpoint candidates:')
for p in all_ckpts[:20]: print(p, f'{p.stat().st_size/1024/1024:.1f} MB')

if not all_ckpts:
    print('No checkpoint was found in the configured run folders. This cell does not create a fake checkpoint. If training has completed, check RUN_DIR and the training log, then rerun this cell.')
    CKPT = None
    CFG = None
else:
    # Prefer best_model.pth; otherwise use the newest checkpoint.
    best = [p for p in all_ckpts if p.name == 'best_model.pth']
    CKPT = best[0] if best else all_ckpts[0]
    CFG = CKPT.parent / 'config.json'
    print('Using checkpoint:', CKPT)
    print('Using config:', CFG)
    if not CFG.exists():
        # Some runs store config.json at the parent run directory.
        alternatives = list(CKPT.parent.rglob('config.json'))
        if alternatives: CFG = alternatives[0]
    assert CFG.exists(), f'Checkpoint found but config.json is missing near {CKPT}'


## 8. Generate speech from unseen Akan text

The supplied `data/test_sentences.txt` contains unseen prompts. We generate all prompts, save the normalized prompts, and play the WAV files in Colab.

In [ ]:
import shutil, subprocess
if CKPT is None:
    print('Skipping inference because there is no checkpoint to load. Complete/resume training, then rerun checkpoint discovery and this cell.')
else:
    if SAMPLE_DIR.exists(): shutil.rmtree(SAMPLE_DIR)
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    cmd = ['python','scripts/infer.py','--ckpt',str(CKPT),'--config',str(CFG),'--file','data/test_sentences.txt','--out',str(SAMPLE_DIR),'--cuda']
    subprocess.run(cmd, check=True)
    prompts_file = SAMPLE_DIR/'prompts.tsv'
    if not prompts_file.exists():
        raise FileNotFoundError(f'Inference finished but prompts.tsv was not created in {SAMPLE_DIR}')
    prompts = prompts_file.read_text(encoding='utf-8').splitlines()
    print('Generated samples:', len(prompts))
    assert len(prompts) >= 5, 'The assignment requires at least five synthesized examples.'
    from IPython.display import Audio, display
    for line in prompts:
        fn, raw, norm = line.split('\t')
        print(f'[{fn}] {raw} -> {norm}')
        display(Audio(filename=str(SAMPLE_DIR/fn)))


## 9. Objective ASR evaluation (WER/CER)

This evaluates the generated speech using Akan-capable MMS ASR. **Interpret WER/CER cautiously** because ASR errors are mixed with TTS errors. The report should also evaluate real held-out recordings with the same ASR if available.

In [ ]:
if CKPT is None or not list(SAMPLE_DIR.glob('*.wav')):
    print('Skipping ASR evaluation because synthesized WAV files are not available yet.')
else:
    subprocess.run(['python','scripts/evaluate.py','asr','--samples',str(SAMPLE_DIR),'--out',str(EVAL_DIR/'asr_results.csv')], check=True)
    import pandas as pd
    asr_df = pd.read_csv(EVAL_DIR/'asr_results.csv')
    display(asr_df)
    print('Mean WER:', asr_df.wer.mean())
    print('Mean CER:', asr_df.cer.mean())


## 10. Listener/MOS evaluation sheet

The generated CSV is randomized so listeners do not see the original prompt order. Distribute the WAV samples plus this sheet to Twi-speaking listeners. Record naturalness, intelligibility, mispronunciations, `ɛ`, `ɔ`, and tone/prosody observations.

After collection, upload the completed sheet to Colab and use the next cell to summarize it.

In [ ]:
if CKPT is None or not list(SAMPLE_DIR.glob('*.wav')):
    print('Skipping listener-sheet creation until inference has produced WAV samples.')
else:
    subprocess.run(['python','scripts/evaluate.py','mos','--samples',str(SAMPLE_DIR),'--out',str(EVAL_DIR/'mos_sheet.csv')], check=True)
    print((EVAL_DIR/'mos_sheet.csv').read_text(encoding='utf-8')[:3000])

# Optional: summarize a completed listener sheet. Set MOS_COMPLETED to its path after collecting ratings.
MOS_COMPLETED = ''
if MOS_COMPLETED:
    mos_df = pd.read_csv(MOS_COMPLETED)
    for col in ['naturalness_1to5','intelligibility_1to5']:
        mos_df[col] = pd.to_numeric(mos_df[col], errors='coerce')
    print(mos_df[['naturalness_1to5','intelligibility_1to5']].agg(['count','mean','std']))


## 11. Create the final experiment manifest and submission package

This collects the exact dataset/preprocessing statistics, checkpoint, generated samples, prompts, evaluation results, and environment information. The report and presentation can then be populated from these artifacts.

In [ ]:
import platform, datetime, json, shutil, os, subprocess
if CKPT is None:
    print('Submission package not created because there is no trained checkpoint yet. Finish training and rerun this cell after inference/evaluation.')
else:
    manifest = {
        'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'variety': 'Asante Twi',
        'language_code': 'aka',
        'dataset': CONFIG,
        'data_stats': stats,
        'checkpoint': str(CKPT),
        'config': str(CFG),
        'samples': [p.name for p in sorted(SAMPLE_DIR.glob('*.wav'))],
        'asr_results': str(EVAL_DIR/'asr_results.csv') if (EVAL_DIR/'asr_results.csv').exists() else None,
        'environment': {
            'python': platform.python_version(),
            'torch': torch.__version__,
            'transformers': transformers.__version__,
            'cuda': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
    }
    (PERSIST/'experiment_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    PACKAGE = PERSIST/'submission_package'
    if PACKAGE.exists(): shutil.rmtree(PACKAGE)
    PACKAGE.mkdir(parents=True)
    if SAMPLE_DIR.exists(): shutil.copytree(SAMPLE_DIR, PACKAGE/'samples', dirs_exist_ok=True)
    if EVAL_DIR.exists(): shutil.copytree(EVAL_DIR, PACKAGE/'eval', dirs_exist_ok=True)
    shutil.copy2(DATA_DIR/'stats.json', PACKAGE/'stats.json')
    shutil.copy2(DATA_DIR/'metadata_val.csv', PACKAGE/'metadata_val.csv')
    shutil.copy2(PERSIST/'experiment_manifest.json', PACKAGE/'experiment_manifest.json')

    # Keep the package self-contained, but avoid retaining multiple redundant training checkpoints.
    shutil.copy2(CKPT, PACKAGE/CKPT.name)
    shutil.copy2(CFG, PACKAGE/CFG.name)
    print('Submission package:', PACKAGE)
    print('Drive free after packaging:', f'{shutil.disk_usage("/content/drive/MyDrive").free/1024**3:.2f} GiB')
    for p in PACKAGE.rglob('*'):
        if p.is_file(): print(p.relative_to(PACKAGE), f'{p.stat().st_size/1024/1024:.2f} MB')
